# A biblioteca `requests` (Python)


**Feito por:** _(Herbert Junior)_  
**Numero** _(4)_  
**Turma** _(12ºPI)_  


## 1. Sobre a biblioteca

**O que é:** uma biblioteca de Python que permite ao programa comunicar com sites e APIs através de pedidos HTTP. É conhecida por ser simples de usar.

**Para que serve:** ir buscar dados a uma API (clima, CEP, GitHub...), enviar dados para um servidor e descarregar páginas.

**Como se instala:** no terminal, escreve-se

```
pip install requests
```

**Quando se usa:** sempre que o programa precisa de ir buscar ou enviar informação pela internet.

In [3]:
import requests
print("Versão do requests:", requests.__version__)

Versão do requests: 2.34.2


## 2. Servidor de teste

Para os exemplos funcionarem sempre (mesmo sem internet), a célula abaixo cria um pequeno servidor no próprio computador. Não é preciso perceber este código: serve só para ter um site onde fazer os pedidos. Com um site verdadeiro, o código de `requests` é igual; só muda o endereço.

In [1]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

class Servidor(BaseHTTPRequestHandler):
    def responder(self, codigo, dados):
        corpo = json.dumps(dados, ensure_ascii=False).encode("utf-8")
        self.send_response(codigo)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(corpo)))
        self.end_headers()
        self.wfile.write(corpo)

    def do_GET(self):
        url = urlparse(self.path)
        if url.path == "/aluno":
            self.responder(200, {"nome": "Ana", "idade": 20})
        elif url.path == "/pesquisa":
            q = parse_qs(url.query)["q"][0]
            self.responder(200, {"pesquisaste": q})
        else:
            self.responder(404, {"erro": "não encontrado"})

    def do_POST(self):
        tamanho = int(self.headers["Content-Length"])
        dados = json.loads(self.rfile.read(tamanho))
        self.responder(201, {"recebido": dados})

    def log_message(self, *args):
        pass

servidor = ThreadingHTTPServer(("127.0.0.1", 0), Servidor)
threading.Thread(target=servidor.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{servidor.server_port}"
print("Servidor de teste a funcionar.")

Servidor de teste a funcionar.


## 3. Exemplos

### Exemplo 1: ler dados (GET)

In [5]:
resposta = requests.get(BASE + "/aluno")

print("Código de estado:", resposta.status_code)   # 200 = correu bem
print("Dados:", resposta.json())                    # JSON -> dicionário
print("Nome:", resposta.json()["nome"])

Código de estado: 200
Dados: {'nome': 'Ana', 'idade': 20}
Nome: Ana


### Exemplo 2: pesquisar com parâmetros

In [6]:
resposta = requests.get(BASE + "/pesquisa", params={"q": "python"})

print("Endereço usado:", resposta.url.split("/pesquisa")[1])
print("Resposta:", resposta.json())

Endereço usado: ?q=python
Resposta: {'pesquisaste': 'python'}


### Exemplo 3: enviar dados (POST) e tratar erros

In [5]:
# Enviar dados
resposta = requests.post(BASE + "/aluno", json={"nome": "Rui", "idade": 22})
print("Código de estado:", resposta.status_code)   # 201 = criado
print("Resposta:", resposta.json())

# Tratar um erro (endereço que não existe)
try:
    resposta = requests.get(BASE + "/nao-existe", timeout=5)
    resposta.raise_for_status()                     # dá erro se o código for 4xx ou 5xx
except requests.exceptions.RequestException as erro:
    print("Deu erro:", erro.response.status_code)

Código de estado: 201
Resposta: {'recebido': {'nome': 'Rui', 'idade': 22}}
Deu erro: 404

## 4. Comparação com o `urllib`

O `urllib` já vem com o Python, mas é mais comprido de escrever. Vê o mesmo pedido feito com as duas bibliotecas:

In [6]:
from urllib.request import urlopen

# Com urllib
with urlopen(BASE + "/aluno") as r:
    dados = json.loads(r.read().decode("utf-8"))
print("urllib  :", dados)

# Com requests
print("requests:", requests.get(BASE + "/aluno").json())

urllib  : {'nome': 'Ana', 'idade': 20}
requests: {'nome': 'Ana', 'idade': 20}

| | `requests` | `urllib` |
|---|---|---|
| Instalação | `pip install requests` | Já vem com o Python |
| Código | Curto e fácil de ler | Mais comprido |
| JSON | `.json()` | É preciso converter à mão |
| Parâmetros | `params=` | É preciso montar o endereço à mão |

**Conclusão:** o `requests` é mais simples de usar; o `urllib` só compensa quando não se pode instalar nada.

## 5. Fontes

- Documentação oficial do Requests: https://requests.readthedocs.io/en/latest/
- Guia rápido (Quickstart): https://requests.readthedocs.io/en/latest/user/quickstart/
- Página no PyPI: https://pypi.org/project/requests/
- Documentação do `urllib.request`: https://docs.python.org/3/library/urllib.request.html